# Localización de la *Magnetic Pileup Boundary* de Marte con MAG y SWEA

**Trabajo final — Machine Learning en Geofluidos**  
**Modelo de referencia elegido: V7b | Misión MAVEN**

Este notebook documenta el problema físico, los datos, la arquitectura, el entrenamiento y la evaluación de **V7b**, siguiendo los scripts del repositorio. La versión V14 queda conservada en el notebook original, separado de este trabajo.

**Objetivo:** localizar temporalmente cruces catalogados de la MPB y estudiar las representaciones latentes de los instrumentos MAG y SWEA.

**Alcance:** se conoce previamente que existe una MPB en un entorno de ±30 minutos. Por tanto, se evalúa **localización condicionada**, no detección autónoma de todas las MPB de una órbita.

## 1. Motivación física

Marte no posee un dipolo magnético global comparable al terrestre. Su atmósfera superior y su ionosfera interactúan con el viento solar y generan una **magnetosfera inducida**.

La *Magnetic Pileup Boundary* (MPB) es una frontera entre regiones plasmáticas con características distintas. En términos de las etiquetas aproximadas que utiliza este modelo, distinguimos:

- **Magnetovaina (*magnetosheath*):** región del flujo solar procesado por el *bow shock*.
- **Magnetosfera inducida:** región de interacción con el entorno planetario, asociada con cambios en el campo magnético y en las distribuciones de partículas.

Un salto del módulo del campo magnético no basta para demostrar que ocurrió una MPB; también pueden existir otras estructuras y discontinuidades. Por eso V7b utiliza **MAG y SWEA**, aprende características de las regiones a ambos lados y compara candidatos temporales.

Las clases de región son **etiquetas débiles inferidas del orden entre el bow shock y la MPB catalogados**; no equivalen a una medición independiente de la composición del plasma.

## 2. Datos y ventanas de entrada

Se parte de un catálogo de **569 cruces MPB**; la cobertura temporal disponible en el manifiesto incluye **569 eventos MAG** y **559 SWEA**. Al utilizar los dos instrumentos, V7b trabaja con eventos con cobertura conjunta y descarta ventanas que no superan los controles de calidad.

Para cada instante candidato \(t_c\) se extraen **600 s** de observaciones:

| Instrumento | Variables de entrada | Dimensión típica por ventana | Tratamiento |
|---|---|---|---|
| MAG | \(B_x,B_y,B_z,|\mathbf B|\) | 4 canales × ~600 muestras | Se conserva la cadencia nativa (~1 s) |
| SWEA | Flujo electrónico promediado angularmente | 12 energías × ~18–19 muestras | Cadencia nativa; no interpolación temporal |

En SWEA se utiliza una grilla fija de **12 energías, espaciadas logarítmicamente entre 30 y 300 eV**. La interpolación se realiza únicamente **sobre el eje energético**; antes se aplica una transformación \(\log(1+F)\) al flujo.

En cada minibatch se utiliza *padding* acompañado por una **máscara temporal**. Las pérdidas de reconstrucción solo consideran las muestras válidas. MAG y SWEA se normalizan **por separado**, calculando media y desviación estándar a partir de los datos de entrenamiento internos de cada fold.

**El modelo no recibe los tiempos catalogados, el nombre del grupo ni la posición real de la MPB como variables de entrada.** Los tiempos catalogados sirven para construir etiquetas y seleccionar las regiones de búsqueda.

## 3. Arquitectura de V7b

V7b tiene **dos ramas convolucionales**, una para cada instrumento:

- **MAG:** entrada de 4 canales → convolución Conv1D de 24 canales → **4 bloques residuales** con dilataciones 1, 2, 4 y 8 → salida de **16 canales latentes temporales**.
- **SWEA:** entrada de 12 canales → Conv1D de 24 canales → **2 bloques residuales** con dilataciones 1 y 2 → salida de **16 canales latentes temporales**.

Los vectores latentes no son 16 números por evento: son **16 series de características**, que mantienen el eje temporal de cada instrumento. Cada rama contiene un decoder auxiliar que reconstruye sus señales a partir de esas características.

### Separación entre los dos lados de un candidato

Los mapas latentes se promedian en dos regiones relativas al centro \(t_c\):

\[
\mathbf z_{\rm izq}=\langle \mathbf z(t)\rangle_{t-t_c<-60\ {\rm s}}, \qquad
\mathbf z_{\rm der}=\langle \mathbf z(t)\rangle_{t-t_c>+60\ {\rm s}}.
\]

Se excluyen los 60 s centrales de cada lado para evitar mezclar de forma inmediata ambas regiones. Se concatenan las características promediadas de MAG y SWEA:

\[
\mathbf L=[\mathbf z_{\rm MAG,izq},\mathbf z_{\rm SWEA,izq}]\in\mathbb R^{32},
\quad
\mathbf R=[\mathbf z_{\rm MAG,der},\mathbf z_{\rm SWEA,der}]\in\mathbb R^{32}.
\]

A partir de ellas se calculan dos tipos de predicción:

1. **Región izquierda/derecha:** dos logits que indican compatibilidad con magnetovaina (0) o magnetosfera inducida (1).
2. **Transición MPB:** un logit a partir de \([\mathbf L,\mathbf R,\mathbf R-\mathbf L,|\mathbf R-\mathbf L|]\), un vector de 128 componentes.

Finalmente:

\[
P_{\rm MPB}(t_c)=\sigma\!\left(s_{\rm transición}(t_c)\right).
\]

La salida es una **probabilidad por ventana candidata**; V7b **no** devuelve una tanh de 600 valores y **no** estima el cruce como el cero de una tanh.

**Código de la arquitectura:** [mag_swea_region_model.py](../src/model/v7b.py).

## 4. Construcción de las etiquetas

Las anotaciones del catálogo permiten identificar **candidatos positivos** cuyos centros están a \(-30\), \(0\) y \(+30\) segundos del tiempo MPB conocido.

Los controles negativos se buscan a distancias mayores (a partir de ±600 s), pero **solo** si la ventana completa cumple las restricciones respecto al *bow shock* asociado, otras MPB y otros BS catalogados. Las ventanas ambiguas **no se tratan automáticamente como negativas**.

Para clasificar los lados físicos de cada candidato:

- Si el BS asociado precede temporalmente a la MPB, la orientación esperada es **magnetovaina → magnetosfera inducida**.
- En el orden inverso, la orientación esperada es **magnetosfera inducida → magnetovaina**.

La aproximación depende de la calidad de esas asociaciones y del catálogo disponible. Una frontera ausente del catálogo puede contaminar los controles considerados negativos.

**Código de etiquetado:** [mpb_weak_regions.py](../src/preprocessing/weak_regions.py).

## 5. Función de pérdida y particularidad de V7b

La red se entrena simultáneamente con cuatro objetivos:

\[
\boxed{\mathcal L=
\mathcal L_{\rm transición}
+\lambda_{\rm región}\mathcal L_{\rm región}
+\lambda_{\rm rec}(\mathcal L_{\rm MAG}+\mathcal L_{\rm SWEA})
+\lambda_{\rm rank}\mathcal L_{\rm ranking}}
\]

- **Transición:** entropía cruzada binaria (BCE) sobre la presencia de MPB cerca del centro de la ventana.
- **Regiones:** BCE sobre las clases débiles izquierda y derecha.
- **Reconstrucción:** error cuadrático medio (MSE) en muestras MAG y SWEA válidas.
- **Ranking de V7b:** compara positivos y negativos del **mismo evento**, forzando una mayor puntuación para los positivos.

Para un par positivo–negativo del mismo cruce, el término de ranking es:

\[
\mathcal L_{\rm par}
=\log\!\left(1+\exp(m+s_{\rm neg}-s_{\rm pos})\right),
\]

donde \(s_{\rm pos}\) y \(s_{\rm neg}\) son **logits**, no probabilidades, y \(m\) es un margen (por defecto **0,5**). La pérdida se promedia por evento.

V7b agrega también **minería de negativos difíciles** (*hard-negative mining*): en entrenamiento identifica controles que reciben puntuaciones inesperadamente altas, los recuerda entre épocas y los combina con controles aleatorios seguros. Esta memoria nunca incorpora datos del fold externo de evaluación.

Los pesos predeterminados en el script son \(\lambda_{\rm región}=0.5\) y \(\lambda_{\rm rec}=0.15\). **Para usar V7b se debe especificar \(\lambda_{\rm rank}>0\)**: el valor predeterminado 0 reproduce la versión V7 anterior. La configuración real de un experimento debe verificarse en sus archivos de métricas o checkpoints.

**Código del ranking:** [mpb_event_ranking.py](../src/model/ranking.py).

## 6. Entrenamiento y validación cruzada

La separación principal es una validación cruzada de **4 folds**, agrupados por mes UTC. Todas las ventanas de un mismo evento permanecen en el mismo fold.

| Fold evaluado | Eventos conjuntos de entrenamiento potenciales | Eventos conjuntos de evaluación potenciales |
|---:|---:|---:|
| 1 | 413 | 146 |
| 2 | 420 | 139 |
| 3 | 424 | 135 |
| 4 | 420 | 139 |

Dentro de los meses de entrenamiento se reserva un subconjunto independiente de **monitor** para seleccionar la mejor época mediante *early stopping* y definir el umbral de decisión binaria. Los normalizadores se ajustan **solo con FIT**.

Para V7b, las ventanas positivas y los controles seguros se agrupan por evento; el ranking compara candidatos del mismo evento. Los parámetros de entrenamiento y el número efectivo de eventos deben registrarse para cada fold.

Los 559 cruces con cobertura temporal SWEA declarada son un máximo potencial: algunos eventos pueden no tener suficientes ventanas conjuntas válidas.

**Nota:** el umbral ajustado en monitor sirve para las métricas de clasificación de candidatos; no cambia la definición del instante de MPB, que se calcula por el máximo de probabilidad.

## 7. Localización de una MPB

En la evaluación de cada evento, V7b recorre centros de ventana en un intervalo de **±30 minutos** respecto de la anotación. El paso predeterminado es **32 segundos**. Cada centro define una ventana conjunta MAG+SWEA de 600 segundos.

Para cada centro \(t_c\), la red devuelve \(P_{\rm MPB}(t_c)\). El estimador es:

\[
\boxed{t_{\rm MPB,pred}=\operatorname*{argmax}_{t_c}P_{\rm MPB}(t_c)}
\]

El error para cada evento se calcula como:

\[
\epsilon_i=\left|t_{\rm MPB,pred,i}-t_{\rm MPB,catálogo,i}\right|.
\]

Las métricas principales son mediana del error, percentil 90 y fracción de eventos con \(\epsilon\le60\) s, \(\epsilon\le120\) s y \(\epsilon\le300\) s.

**Precauciones para interpretar los resultados:**

- El cálculo está **condicionado** a una vecindad que contiene una MPB catalogada; no mide detección autónoma sobre toda la órbita.
- El máximo se elige entre centros discretos de la grilla de 32 s. No equivale a una resolución temporal de 1 s.
- El código selecciona el máximo aunque su probabilidad absoluta sea pequeña; el umbral de monitor no impone una opción “sin MPB” para esta evaluación condicionada.
- La precisión, recall y F1 de candidatos utilizan **etiquetas débiles** de los controles y positivos seleccionados, no verdaderos negativos exhaustivos de órbitas completas.

**Código de entrenamiento y evaluación:** [5b_train_mag_swea_regions.py](../scripts/02_training/01_train_v7b.py).

## 8. Resultados y discusión

En esta sección incorporaremos **las métricas reales de V7b** a partir de los archivos de entrenamiento guardados. Conviene conservar por separado:

1. Los cuatro folds, el número de eventos evaluables y la mediana del error.
2. Las fracciones dentro de 1 y 2 minutos, con el denominador explícito.
3. Las métricas de ranking y las regiones etiquetadas débilmente.
4. Las limitaciones del catálogo, los casos sin cobertura y la dependencia de la selección de ventanas.

El notebook aún no contiene números copiados manualmente para no confundir resultados de V7, V7b, V14 y las pruebas MAG-only recientes.

### Próximo análisis: representaciones latentes

Para entender qué aprendió V7b se pueden extraer los 16 canales latentes de MAG y de SWEA antes y después de cada candidato, y estudiar su dependencia con observables físicos. Los espacios de distintos folds fueron entrenados de forma independiente y **no deben mezclarse como si compartieran ejes latentes idénticos**.

In [ ]:
# Lectura opcional de un resumen V7b si el archivo ya está disponible localmente.
# No entrena el modelo ni descarga datos. En Colab, primero montar Drive o subir
# el archivo de métricas y actualizar la ruta.
from pathlib import Path
import json

metrics_path = Path("results/metrics/mag_swea_regions_ranking_v1/cv_summary.json")
if metrics_path.exists():
    summary = json.loads(metrics_path.read_text(encoding="utf-8"))
    print("Arquitectura:", summary.get("architecture"))
    print("Folds completados:", summary.get("completed_folds"))
    for fold in summary.get("fold_results", []):
        print(
            f"Fold {fold.get('fold')}: "
            f"evaluables={fold.get('val_evaluable_events')}/{fold.get('val_total_events')}, "
            f"mediana={fold.get('median_abs_peak_error_s')} s, "
            f"<=60 s: {fold.get('fraction_within_60s')}"
        )
else:
    print("Todavía no se encontró el resumen V7b en:", metrics_path)
    print("Subir el archivo o ajustar metrics_path a su ubicación real.")